All the required libraries are imported.

In [1]:
import numpy as np
import pandas as pd
import urllib
import math
import statistics as st
from sklearn import preprocessing

import requests
from pandas import json_normalize
import json

from geopy.geocoders import Nominatim

import folium
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

The name of the town/city and the type of business or recreational venue is taken as input. The user can select a business type and has a list of options to choose from.

In [2]:
entry=input("Enter name of city/town along with state or country : ")

Enter name of city/town along with state or country : San Jose, California


In [3]:
categories=pd.read_csv('/content/categories.csv')
categories=categories.sort_values('categories')

In [4]:
for i in categories['categories'].values:
    print(i)

Adult Boutique
Afghan Restaurant
African Restaurant
American Restaurant
Amphitheater
Andhra Restaurant
Aquarium
Arcade
Argentinian Restaurant
Art Gallery
Arts & Crafts Store
Arts & Entertainment
Asian Restaurant
Athletics & Sports
Awadhi Restaurant
BBQ Joint
Bagel Shop
Bakery
Bar
Baseball Stadium
Basketball Stadium
Bath House
Beach
Bed & Breakfast
Beer Bar
Beer Store
Beijing Restaurant
Bengali Restaurant
Big Box Store
Bike Shop
Bistro
Boat or Ferry
Bookstore
Boutique
Brazilian Restaurant
Breakfast Spot
Brewery
Bridge
Bubble Tea Shop
Buddhist Temple
Building
Burger Joint
Butcher
Cafeteria
Café
Candy Store
Cantonese Restaurant
Capitol Building
Chaat Place
Cheese Shop
Chinese Restaurant
Chocolate Shop
Church
Clothing Store
Club House
Cocktail Bar
Coffee Shop
Comic Shop
Concert Hall
Construction & Landscaping
Convenience Store
Convention Center
Cosmetics Shop
Cricket Ground
Cultural Center
Cupcake Shop
Cycle Studio
Dance Studio
Deli , Bodega
Dentist's Office
Department Store
Dessert Shop
D

In [5]:
business=input('Select the type of business you are currently planning to open. Choose from the list given above : ')

Select the type of business you are currently planning to open. Choose from the list given above : Restaurant


The required parameters are arranged to make a Foursquare API call.

In [6]:
address = entry

geolocator = Nominatim(user_agent="coursera-capstone-project")
location = geolocator.geocode(address)
latitude = location.latitude
longitude = location.longitude

In [7]:
CLIENT_ID = 'SWWNHHEKWCPEHJKBPCWLGGN5BNKHHKU243TGGEY3MHO4ZK4H'
CLIENT_SECRET = '4RUFRQHM00UUVURJIIMO4ITB2V12C1NX1Z3KTWTORWW0AB5Z'
VERSION = '20180604'
LIMIT = 100
radius=5000
url = 'https://api.foursquare.com/v2/venues/explore?&client_id={}&client_secret={}&v={}&ll={},{}&radius={}&limit={}'.format(
    CLIENT_ID,
    CLIENT_SECRET,
    VERSION,
    latitude,
    longitude,
    radius,
    LIMIT)
results = requests.get(url).json()
print(results)

{'meta': {'code': 200, 'requestId': '672ecdecc5db2b54563d62f2'}, 'response': {'queryRefinements': {'target': {'type': 'path', 'url': '/venue/explore', 'params': {'ll': '37.336166,-121.890591', 'radius': '5000'}}, 'refinements': [{'query': 'Food'}, {'query': 'Nightlife'}, {'query': 'Coffee'}, {'query': 'Shops'}, {'query': 'Arts'}, {'query': 'Outdoors'}]}, 'suggestedFilters': {'header': 'Tap to show:', 'filters': [{'name': '$-$$$$', 'key': 'price'}, {'name': 'Open now', 'key': 'openNow'}]}, 'headerLocation': 'San Jose', 'headerFullLocation': 'San Jose', 'headerLocationGranularity': 'city', 'totalResults': 208, 'suggestedBounds': {'ne': {'lat': 37.38116634500005, 'lng': -121.83409930846796}, 'sw': {'lat': 37.29116625499996, 'lng': -121.94708269153205}}, 'groups': [{'type': 'Recommended Places', 'name': 'recommended', 'items': [{'reasons': {'count': 0, 'items': [{'summary': 'This spot is popular', 'type': 'general', 'reasonName': 'globalInteractionReason'}]}, 'venue': {'id': '4e6750db15200

The call returns a json file which is stored in a variable.

Only the 'items' are taken from the json file.

In [8]:
items = results['response']['groups'][0]['items']

The json file is normalized and converted into a dataframe.

In [9]:
dataframe = json_normalize(items)
dataframe.head()

,referralId,reasons.count,reasons.items,venue.id,venue.name,venue.location.address,venue.location.crossStreet,venue.location.lat,venue.location.lng,venue.location.labeledLatLngs,...,venue.venuePage.id,venue.createdAt,photo.id,photo.createdAt,photo.prefix,photo.suffix,photo.width,photo.height,photo.visibility,venue.deliveryProviders
0,e-0-4e6750db152001e1f7122a11-0,0,"[{'summary': 'This spot is popular', 'type': '...",4e6750db152001e1f7122a11,San Pedro Square Market,87 N San Pedro St,at W St John St,37.335648,-121.893340,"[{'label': 'display', 'lat': 37.335648, 'lng':...",...,91053438,1315393755,4fa72665e4b0c1088f93bf4f,1336354405,https://fastly.4sqi.net/img/general/,/z8yM1eV4F7-5ov8cxnBGrhrt_f_OttKtHNScfjDMLF8.jpg,612,612,public,NaN
1,e-0-5293dcf9498ee4056f1ecc87-1,0,"[{'summary': 'This spot is popular', 'type': '...",5293dcf9498ee4056f1ecc87,Ike's Sandwiches,"75 E Santa Clara St, Ste 130",at N 2nd St,37.336852,-121.889436,"[{'label': 'display', 'lat': 37.33685220939853...",...,NaN,1385422073,540e1778498e50365614597c,1410209656,https://fastly.4sqi.net/img/general/,/12862390_KLNoi124TTPmjG1e8TDWWKBlN6EFqxxTsC84...,640,640,public,"[{'id': '342548', 'url': 'https://www.grubhub...."
2,e-0-53336ce5498e6ec45340b722-2,0,"[{'summary': 'This spot is popular', 'type': '...",53336ce5498e6ec45340b722,Paper Plane,72 S 1st St,at Post St,37.334921,-121.889579,"[{'label': 'display', 'lat': 37.33492113763415...",...,NaN,1395879141,555fbc73498e3997a3bded64,1432337523,https://fastly.4sqi.net/img/general/,/17061080_89qx8wv4SqtrPCsXc7oY97egLFHOSrl3qvua...,1920,1440,public,NaN
3,e-0-4abbbdd9f964a520a38420e3-3,0,"[{'summary': 'This spot is popular', 'type': '...",4abbbdd9f964a520a38420e3,Freshly Baked Eatery,152 N 3rd St,btwn E St James & E St John St,37.339404,-121.890051,"[{'label': 'display', 'lat': 37.33940424234091...",...,NaN,1253817817,4f95a2b0e4b0ab5f0b927ef8,1335206576,https://fastly.4sqi.net/img/general/,/LKzerlc_VB6XnXVcxwzqGPHvy8guIG-I6QcCaYzh35E.jpg,612,612,public,NaN
4,e-0-4a6baaf9f964a5208dcf1fe3-4,0,"[{'summary': 'This spot is popular', 'type': '...",4a6baaf9f964a5208dcf1fe3,Silicon Valley Capital Club,50 W San Fernando St,NaN,37.333789,-121.889213,"[{'label': 'entrance', 'lat': 37.334059, 'lng'...",...,NaN,1248570105,5569399c498e7f3ff62d8d58,1432959388,https://fastly.4sqi.net/img/general/,/59495_ptnnVumPvRis2_XSa2fhhb2iReWgrf9aY0jn8Mq...,640,640,public,NaN


The required columns are retrieved and renamed as required.

In [10]:
def get_category_type(row):
    try:
        categories_list = row['categories']
    except:
        categories_list = row['venue.categories']

    if len(categories_list) == 0:
        return None
    else:
        return categories_list[0]['name']

In [11]:
venues = results['response']['groups'][0]['items']

nearby_venues = json_normalize(venues)

filtered_columns = ['venue.name', 'venue.categories', 'venue.location.lat', 'venue.location.lng']
nearby_venues =nearby_venues.loc[:, filtered_columns]

nearby_venues['venue.categories'] = nearby_venues.apply(get_category_type, axis=1)

nearby_venues.columns = [col.split(".")[-1] for col in nearby_venues.columns]

nearby_venues['categories']=nearby_venues['categories'].str.replace('/',',')

old_nearby_venues=nearby_venues

In [12]:
check_list=['Restaurant','Store','Shop','Garden','Park','Gym','Museum','Theatre','Hotel','Cafe','Market','Pub']
new_check_list=['Restaurant','Everyday Store','Everyday Shop','Garden','Park','Gym','Museum','Theatre','Hotel','Cafe','Market','Pub']

Data cleaning is thoroughly performed. Some venue categories are renamed to some convenient titles which would further make it easy and effective when the machine learning algorithms are applied. Two copies of the same dataframe is made but with slight variations. The stores,shops,bars and restaurants are not renamed for the dataframe 'old_nearby_venues' so as to enable the user in giving an accurate input later.

In [13]:
new=nearby_venues[~nearby_venues['categories'].str.contains('Restaurant')]
new=new[~new['categories'].str.contains('Bar')]
new=new[~new['categories'].str.contains('Plaza')]
new=new[~new['categories'].str.contains('Store')]
new=new[~new['categories'].str.contains('store')]
new=new[~new['categories'].str.contains('Shop')]
new=new[~new['categories'].str.contains('shop')]
new=new[~new['categories'].str.contains('Garden')]
new=new[~new['categories'].str.contains('Park')]
new=new[~new['categories'].str.contains('Gym')]
new=new[~new['categories'].str.contains('Museum')]
new=new[~new['categories'].str.contains('Theatre')]
new=new[~new['categories'].str.contains('Theater')]
new=new[~new['categories'].str.contains('Hotel')]
new=new[~new['categories'].str.contains('Café')]
new=new[~new['categories'].str.contains('Market')]
new=new[~new['categories'].str.contains('market')]
new=new[~new['categories'].str.contains('Pub')]
new=new[~new['categories'].str.contains('pub')]

In [14]:
res=nearby_venues[nearby_venues['categories'].str.contains('Restaurant')]
bar=nearby_venues[nearby_venues['categories'].str.contains('Bar')]
plaza=nearby_venues[nearby_venues['categories'].str.contains('Plaza')]
store=nearby_venues[nearby_venues['categories'].str.contains('Store')]
store1=nearby_venues[nearby_venues['categories'].str.contains('store')]
shop=nearby_venues[nearby_venues['categories'].str.contains('Shop')]
shop1=nearby_venues[nearby_venues['categories'].str.contains('shop')]
gar=nearby_venues[nearby_venues['categories'].str.contains('Garden')]
park=nearby_venues[nearby_venues['categories'].str.contains('Park')]
gym=nearby_venues[nearby_venues['categories'].str.contains('Gym')]
mus=nearby_venues[nearby_venues['categories'].str.contains('Museum')]
thr=nearby_venues[nearby_venues['categories'].str.contains('Theatre')]
thr1=nearby_venues[nearby_venues['categories'].str.contains('Theater')]
hot=nearby_venues[nearby_venues['categories'].str.contains('Hotel')]
cafe=nearby_venues[nearby_venues['categories'].str.contains('Café')]
mark=nearby_venues[nearby_venues['categories'].str.contains('Market')]
mark1=nearby_venues[nearby_venues['categories'].str.contains('market')]
pub=nearby_venues[nearby_venues['categories'].str.contains('Pub')]
pub1=nearby_venues[nearby_venues['categories'].str.contains('pub')]

res.categories='Restaurant'
bar.categories='Bar'
plaza.categories='Plaza'
store.categories='Everyday Store'
store1.categories='Everyday Store'
shop.categories='Everyday Shop'
shop1.categories='Everyday Shop'
gar.categories='Garden'
park.categories='Park'
gym.categories='Gym'
mus.categories='Museum'
thr.categories='Theatre'
thr1.categories='Theatre'
hot.categories='Hotel'
cafe.categories='Cafe'
mark.categories='Market'
mark1.categories='Market'
pub.categories='Pub'
pub1.categories='Pub'

<ipython-input-14-bab510b030e5>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  res.categories='Restaurant'
<ipython-input-14-bab510b030e5>:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bar.categories='Bar'
<ipython-input-14-bab510b030e5>:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#return

The renamed column data is appended to the new dataframe named 'new' and is finally stored in the existing dataframe 'nearby_venues'.

In [15]:
import pandas as pd
new = pd.concat([new, res, bar, plaza, store, store1, shop, shop1, gar, park, gym, mus, thr, thr1, hot, cafe, mark, mark1, pub, pub1])

In [16]:
nearby_venues=new

In [17]:
new=old_nearby_venues[~old_nearby_venues['categories'].str.contains('Garden')]
new=new[~new['categories'].str.contains('Plaza')]
new=new[~new['categories'].str.contains('Park')]
new=new[~new['categories'].str.contains('Gym')]
new=new[~new['categories'].str.contains('Museum')]
new=new[~new['categories'].str.contains('Theatre')]
new=new[~new['categories'].str.contains('Theater')]
new=new[~new['categories'].str.contains('Hotel')]
new=new[~new['categories'].str.contains('Café')]
new=new[~new['categories'].str.contains('Market')]
new=new[~new['categories'].str.contains('market')]
new=new[~new['categories'].str.contains('Pub')]
new=new[~new['categories'].str.contains('pub')]

In [18]:
gar=old_nearby_venues[old_nearby_venues['categories'].str.contains('Garden')]
plaza=old_nearby_venues[old_nearby_venues['categories'].str.contains('Plaza')]
park=old_nearby_venues[old_nearby_venues['categories'].str.contains('Park')]
gym=old_nearby_venues[old_nearby_venues['categories'].str.contains('Gym')]
mus=old_nearby_venues[old_nearby_venues['categories'].str.contains('Museum')]
thr=old_nearby_venues[old_nearby_venues['categories'].str.contains('Theatre')]
thr1=old_nearby_venues[old_nearby_venues['categories'].str.contains('Theater')]
hot=old_nearby_venues[old_nearby_venues['categories'].str.contains('Hotel')]
cafe=old_nearby_venues[old_nearby_venues['categories'].str.contains('Café')]
mark=old_nearby_venues[old_nearby_venues['categories'].str.contains('Market')]
mark1=old_nearby_venues[old_nearby_venues['categories'].str.contains('market')]
pub=old_nearby_venues[old_nearby_venues['categories'].str.contains('Pub')]
pub1=old_nearby_venues[old_nearby_venues['categories'].str.contains('pub')]

In [19]:
gar.categories='Garden'
plaza.categories='Plaza'
park.categories='Park'
gym.categories='Gym'
mus.categories='Museum'
thr.categories='Theatre'
thr1.categories='Theatre'
hot.categories='Hotel'
cafe.categories='Café'
mark.categories='Market'
mark1.categories='Market'
pub.categories='Pub'
pub1.categories='Pub'

<ipython-input-19-34e71b2bc628>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gar.categories='Garden'
<ipython-input-19-34e71b2bc628>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  plaza.categories='Plaza'
<ipython-input-19-34e71b2bc628>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning

The renamed column data is appended to the new dataframe named 'new' and is finally stored in the existing dataframe 'old_nearby_venues'.

In [20]:
import pandas as pd

new = pd.concat([new, gar, plaza, park, gym, mus, thr, thr1, hot, cafe, mark, mark1, pub, pub1])

In [21]:
old_nearby_venues=new

The 'categories' column data is split into two strings with the delimiter being comma(,), and only the first string is kept.

In [22]:
old_nearby_venues['categories'] = old_nearby_venues['categories'].str.split(',').str[0]

In [23]:
nearby_venues['categories'] = nearby_venues['categories'].str.split(',').str[0]

White spaces from the ends of string are removed.

In [24]:
old_nearby_venues.categories = old_nearby_venues.categories.str.rstrip()

In [25]:
row_count = nearby_venues.shape[0]
row_count

100

One hot encoding is performed on the 'categories' column.

In [26]:
onehot = pd.get_dummies(nearby_venues[['categories']], prefix="", prefix_sep="")

DBSCAN code on the one hot encoder output

In [28]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# Assuming 'onehot' DataFrame is already created with one-hot encoded category data.
# Standardize the data for DBSCAN, as it works better with scaled data.
scaler = StandardScaler()
scaled_data = scaler.fit_transform(onehot)

# Set parameters for DBSCAN (these may need tuning based on your data).
eps = 0.5         # maximum distance between samples in a neighborhood
min_samples = 5   # minimum number of samples in a neighborhood for a core point

# Perform DBSCAN clustering.
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(scaled_data)

# Add the DBSCAN labels to the original DataFrame for easy reference.
onehot['DBSCAN_Cluster'] = dbscan_labels

# Display the clustering result.
print(onehot['DBSCAN_Cluster'].value_counts())
print(onehot.head())

DBSCAN_Cluster
-1    42
 0    28
 3    14
 1     9
 2     7
Name: count, dtype: int64
    Art Gallery  BBQ Joint  Bakery    Bar  Breakfast Spot  Brewery  \
0         False      False   False  False           False    False   
1         False      False   False  False           False    False   
3         False      False   False  False           False    False   
7         False      False    True  False           False    False   
12        False      False   False  False           False    False   

    Burger Joint   Cafe   Deli  Everyday Shop  ...   Park  \
0          False  False  False          False  ...  False   
1          False  False  False          False  ...  False   
3          False  False  False          False  ...  False   
7          False  False  False          False  ...  False   
12         False  False  False          False  ...  False   

    Performing Arts Venue  Pizzeria  Playground  Plaza  Restaurant  \
0                   False     False       False  False  

The cluster labels are inserted into the dataframe 'nearby_venues' in a column named 'cluster'.

In [29]:
# Insert DBSCAN cluster labels as a new column in nearby_venues DataFrame
nearby_venues.insert(0, 'dbscan_cluster', dbscan_labels)

A folium map is generated displaying all the venue categories clustered in different colors.

In [30]:
import folium
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as colors

# Assuming 'latitude' and 'longitude' are defined for the map center
# Create map
map_clusters_dbscan = folium.Map(location=[latitude, longitude], zoom_start=11)

# Set color scheme for DBSCAN clusters
unique_clusters = onehot['DBSCAN_Cluster'].unique()
n_clusters = len(unique_clusters)
colors_array = cm.rainbow(np.linspace(0, 1, n_clusters))
rainbow = [colors.rgb2hex(i) for i in colors_array]

# Add markers to the map
for lat, lon, name, cluster in zip(nearby_venues['lat'], nearby_venues['lng'], nearby_venues['name'], onehot['DBSCAN_Cluster']):
    label = folium.Popup(f'Cluster {cluster}: {name}', parse_html=True)
    cluster_color = rainbow[cluster] if cluster != -1 else "#000000"  # Black color for noise (-1)
    folium.CircleMarker(
        [lat, lon],
        radius=5,
        popup=label,
        color=cluster_color,
        fill=True,
        fill_color=cluster_color,
        fill_opacity=0.7
    ).add_to(map_clusters_dbscan)

# Prepare cluster analysis summary
avg_lat_dbscan = []
avg_lng_dbscan = []
clustered_name_dbscan = []
clustered_percent_dbscan = []

for cluster in unique_clusters:
    cust = nearby_venues[onehot['DBSCAN_Cluster'] == cluster]
    if cust.shape[0] == 0:
        continue

    if len(cust['categories'].unique()) == 1:
        count = cust.shape[0]
        mean_lat = cust['lat'].mean()
        mean_lng = cust['lng'].mean()
        avg_lat_dbscan.append(mean_lat)
        avg_lng_dbscan.append(mean_lng)
        val = cust['categories'].values[0]
        n = len(cust)
        percent = (n / row_count) * 100
        clustered_name_dbscan.append(val)
        clustered_percent_dbscan.append(round(percent))
    else:
        mean_lat = cust['lat'].mean()
        mean_lng = cust['lng'].mean()
        avg_lat_dbscan.append(mean_lat)
        avg_lng_dbscan.append(mean_lng)
        n = len(cust)
        percent = (n / row_count) * 100
        clustered_name_dbscan.append("Miscellaneous")
        clustered_percent_dbscan.append(round(percent))

# Display the map
map_clusters_dbscan

The major venue categories along with their percentages are taken into two different lists. Also, the average latitudes and longitudes of the major categories are stored in other lists.

In [31]:
# Initialize lists to store DBSCAN cluster analysis results
avg_lat_dbscan = []
avg_lng_dbscan = []
clustered_name_dbscan = []
clustered_percent_dbscan = []

# Iterate over each unique cluster in DBSCAN (including noise, labeled as -1)
for cluster in nearby_venues['dbscan_cluster'].unique():
    cust = nearby_venues[nearby_venues['dbscan_cluster'] == cluster]
    cust = pd.DataFrame(cust)

    if len(cust['categories'].unique()) == 1:
        # If the cluster contains only one unique category
        count = cust.shape[0]
        mean_lat = cust['lat'].mean()
        mean_lng = cust['lng'].mean()
        avg_lat_dbscan.append(mean_lat)
        avg_lng_dbscan.append(mean_lng)
        val = cust['categories'].values[0]
        n = len(cust)
        percent = (n / row_count) * 100
        percent = round(percent)
        clustered_name_dbscan.append(val)
        clustered_percent_dbscan.append(percent)
    else:
        # If the cluster contains multiple categories
        mean_lat = cust['lat'].mean()
        mean_lng = cust['lng'].mean()
        avg_lat_dbscan.append(mean_lat)
        avg_lng_dbscan.append(mean_lng)
        n = len(cust)
        percent = (n / row_count) * 100
        percent = round(percent)
        clustered_name_dbscan.append("Miscellaneous")
        clustered_percent_dbscan.append(percent)

# Result lists contain DBSCAN clustering details
# avg_lat_dbscan, avg_lng_dbscan: Average latitudes and longitudes for each DBSCAN cluster
# clustered_name_dbscan: Category name or "Miscellaneous" if multiple categories
# clustered_percent_dbscan: Percentage of venues in each DBSCAN cluster

The number of venue categories the user would love to have around his/her business is taken as input. Also, the venue categories of that particular town/city is displayed for the user to make a choice from.

In [32]:
options=[]
c=0
df=pd.DataFrame(columns=['categories'])
df.categories=old_nearby_venues['categories']
df=df.categories.unique()
df=pd.DataFrame(df,columns=['categories'])
df=df.sort_values('categories')
for i in df['categories'].values:
    print(i)
    c=c+1
print()
maximum=input("Enter number of venues you would love to have around your business from the list above. You can enter a maximum of " + (str)(c) + " venues. ")

Art Gallery
BBQ Joint
Bagel Shop
Bakery
Bar
Beer Bar
Beer Store
Breakfast Spot
Brewery
Bubble Tea Shop
Burger Joint
Café
Cajun and Creole Restaurant
Caribbean Restaurant
Chinese Restaurant
Cocktail Bar
Coffee Shop
Deli
Dessert Shop
Dive Bar
Ethiopian Restaurant
Food Court
Frozen Yogurt Shop
Garden
German Restaurant
Gift Store
Greek Restaurant
Grocery Store
Hiking Trail
Hockey Stadium
Hotel
Italian Restaurant
Japanese Restaurant
Mexican Restaurant
Museum
Music Venue
Neighborhood
Office
Opera House
Park
Performing Arts Venue
Pet Supplies Store
Pizzeria
Playground
Plaza
Restaurant
Sandwich Spot
Seafood Restaurant
Smoke Shop
Sushi Restaurant
Taco Restaurant
Theatre
Vegan and Vegetarian Restaurant
Vietnamese Restaurant
Wine Bar
Zoo

Enter number of venues you would love to have around your business from the list above. You can enter a maximum of 56 venues. 3


In [33]:
for i in range(0,(int)(maximum)):
    option=input('Option ' + (str)(i+1) + ' : ')
    options.append(option)

Option 1 : Restaurant
Option 2 : Wine Shop
Option 3 : Office


The new possible coordinates are calculated based on the information given by the user.

In [34]:
opt_lat = []
opt_lng = []
lat = 0
lng = 0
count = 0

for i in range(len(options)):
    check = options[i]
    for (j, k, l) in zip(old_nearby_venues['categories'].values, old_nearby_venues['lat'].values, old_nearby_venues['lng'].values):
        if j == check:
            count += 1
            lat += k
            lng += l

    # Check to avoid division by zero
    if count > 0:
        lat = lat / count
        lng = lng / count
    else:
        lat, lng = None, None  # or use some default values if no match is found

    opt_lat.append(lat)
    opt_lng.append(lng)

    # Reset values for the next iteration
    lat = 0
    lng = 0
    count = 0


In [35]:

import statistics as st

# Filter out None values before calculating the mean
valid_opt_lat = [x for x in opt_lat if x is not None]
valid_opt_lng = [x for x in opt_lng if x is not None]

# Now calculate the mean
if valid_opt_lat and valid_opt_lng:  # Ensure there are valid values to calculate the mean
    lat = st.mean(valid_opt_lat)
    lng = st.mean(valid_opt_lng)
else:
    lat, lng = None, None  # Handle the case where there are no valid values

In [36]:
result=-1
j=0
for i in check_list:
    result=business.find(i)
    if result!=-1:
        business=new_check_list[j]
    j=j+1

The support vector machine classification algorithm is applied to predict the category of the business or venue given as input by the user.

In [39]:
import statistics as st
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Initialize variables
final = 0  # Set final to 0 initially
valid_opt_lat = [x for x in opt_lat if x is not None]
valid_opt_lng = [x for x in opt_lng if x is not None]

# Step 1: Calculate mean latitude and longitude if there are valid values
if valid_opt_lat and valid_opt_lng:
    lat = st.mean(valid_opt_lat)
    lng = st.mean(valid_opt_lng)
else:
    lat, lng = None, None

# Step 2: Update business category if found in check_list
result = -1
for j, item in enumerate(check_list):
    result = business.find(item)
    if result != -1:
        business = new_check_list[j]
        break

# Step 3: Prepare dummy-encoded features and apply DBSCAN
feature = pd.get_dummies(nearby_venues['categories'])
all_columns = feature.columns  # Store all dummy column names for consistency

# Standardize and apply DBSCAN
x = StandardScaler().fit_transform(feature)
dbscan = DBSCAN(eps=0.5, min_samples=5)  # Adjust parameters as needed
dbscan_labels = dbscan.fit_predict(x)

# Add DBSCAN labels to the DataFrame
nearby_venues['dbscan_cluster'] = dbscan_labels

# Step 4: Train a Random Forest Classifier on DBSCAN clusters (excluding noise)
filtered_data = nearby_venues[nearby_venues['dbscan_cluster'] != -1]
filtered_features = pd.get_dummies(filtered_data['categories']).reindex(columns=all_columns, fill_value=0)
x_filtered = StandardScaler().fit_transform(filtered_features)
filtered_labels = filtered_data['dbscan_cluster']

# Train-test split
x_train, x_test, y_train, y_test = train_test_split(x_filtered, filtered_labels, test_size=0.2, random_state=4)

# Train the Random Forest Classifier
rf_classifier = RandomForestClassifier(random_state=4)
rf_classifier.fit(x_train, y_train)

# Step 5: Check if the business category exists in nearby_venues
flag = 0
predicted_cluster = -1
for category in nearby_venues['categories'].values:
    if category == business:
        flag = 1
        break

# Step 6: If the business category exists, append the new row and predict with Random Forest
if flag == 1:
    # Create a new row with the business info
    new_row = pd.DataFrame([(None, None, business, lat, lng)], columns=['cluster', 'name', 'categories', 'lat', 'lng'])
    nearby_venues = pd.concat([nearby_venues, new_row], ignore_index=True)

    # Re-create dummy-encoded features with aligned columns
    feature = pd.get_dummies(nearby_venues['categories']).reindex(columns=all_columns, fill_value=0)
    x = StandardScaler().fit_transform(feature)

    # Predict the cluster for the new row
    rf_predict = rf_classifier.predict(x)
    predicted_cluster = int(rf_predict[-1])  # Get the prediction for the last (new) entry

# Step 7: Assign to default cluster if `final == 1` and business category not found
if final == 1 and flag == 0:
    predicted_cluster = 0

print("Predicted DBSCAN cluster:", predicted_cluster)

Predicted DBSCAN cluster: 0


Required markers are added to the folium map i.e the new coordinates.

In [40]:
folium.Marker(
    location=[lat,lng],
    icon=folium.Icon(icon='cloud')
).add_to(map_clusters_dbscan)

The result is a folium map with the popup being the required coordinates of the location where the business or the venue could be possibly set up, along with the following information derived from various input taken from the user.

In [41]:
print('Major categories in ' + entry + ' are -->')
print()
for (i,j) in (zip(clustered_name_dbscan,clustered_percent_dbscan)):
    print((str)(i) + ' : ' + (str)(j) + '%')
print()
if len(clustered_name_dbscan) > 1:
    if flag==0 and final==0:
        print('Your business does not fall into any category we have made. This is a new type of business you are trying to open which is almost not present in the location you have provided. This significantly boosts the chances of you being successful in your business. Based on the data you have provided to us, we recommend you to set up your business within 1 km radius of the popup marked on the map given below.')
    else:
        print('The kind of business you are trying to open falls into the ' + (str)(clustered_percent_dbscan[predicted_cluster]) + '% ' + (str)(clustered_name_dbscan[predicted_cluster]) + ' category. Based on the data you have provided to us, we recommend you to set up your business within 1 km radius of the popup marked on the map given below.')
    print('Popup coordinates --> Latitude : ' + (str)(lat) + '  Longitude : ' + (str)(lng))
if len(clustered_name_dbscan) == 1:
    print('The kind of business you are trying to open falls into the one and only ' + (str)(clustered_name_dbscan[0]) + ' category. Based on the data you have provided to us, we recommend you to set up your business within 1 km radius of the popup marked on the map given below.')
    print('Popup coordinates --> Latitude : ' + (str)(lat) + '  Longitude : ' + (str)(lng))
map_clusters_dbscan

Major categories in San Jose, California are -->

Miscellaneous : 42%
Restaurant : 28%
Bar : 9%
Everyday Store : 7%
Everyday Shop : 14%

The kind of business you are trying to open falls into the 42% Miscellaneous category. Based on the data you have provided to us, we recommend you to set up your business within 1 km radius of the popup marked on the map given below.
Popup coordinates --> Latitude : 37.325131640333765  Longitude : -121.88197261787954
